# День 1 · exponential backoff

Сетевые сбои, `429 Too Many Requests` и `5xx` от провайдера — норма, а не авария: клиент обязан их
переживать сам. Но повторять стоит не любую ошибку — `400`/`404` (неверная модель, плохой запрос)
от повтора не изменятся, это баг в запросе, а не временная неполадка.

In [ ]:
import random
import sys
import time

import labkit  # noqa: F401  читает .env
from openai import APIConnectionError, APITimeoutError, InternalServerError, RateLimitError   # ошибки, после которых есть смысл повторить

from client import make_client, MODEL

# --- НАСТРОЙКИ ---
TIMEOUT = 60.0      # секунд на запрос; поставь 0.3, чтобы увидеть ретраи по таймауту
ATTEMPTS = 5        # максимум попыток
BASE_DELAY = 1.0    # первая пауза, дальше удвоение: 1, 2, 4, 8…
MAX_DELAY = 20.0    # потолок паузы
PROMPT = "Одним предложением: зачем клиенту к LLM нужны ретраи?"

RETRYABLE = (RateLimitError, APIConnectionError, APITimeoutError, InternalServerError)   # 429, сеть, таймаут, 5xx

## Механизм на числах, без сети

Вот и весь backoff — пауза удваивается на каждой попытке (`base * 2 ** (attempt - 1)`) до потолка
`cap`, плюс случайная добавка (джиттер), чтобы много клиентов, поймавших одну и ту же ошибку
одновременно, не ударили по провайдеру снова синхронно, все разом, ровно через одну и ту же паузу.
Ниже — та же формула, что в `with_retries`, но выполненная без единого сетевого вызова, только
чтобы увидеть последовательность пауз сразу, а не ждать реального сбоя:

In [ ]:
for attempt in range(1, ATTEMPTS + 1):
    delay = min(MAX_DELAY, BASE_DELAY * 2 ** (attempt - 1))
    print(f"попытка {attempt}: базовая пауза {delay:.1f}s (+ джиттер 0–0.5s)")

## Тот же механизм на настоящем запросе

`with_retries` оборачивает любой вызов: если он бросает одну из `RETRYABLE` ошибок — печатает,
сколько ждёт, спит и пробует снова; на последней попытке отдаёт исключение наверх, а не проглатывает
его. В обычной работе (без сбоев) ты просто не увидишь строк про паузу — ответ придёт с первой
попытки, и это тоже нормальный, ожидаемый результат.

In [ ]:
def with_retries(fn, attempts: int = ATTEMPTS, base: float = BASE_DELAY, cap: float = MAX_DELAY):
    """Вызывает fn(); при временной ошибке ждёт и повторяет. Exponential backoff с джиттером."""
    for attempt in range(1, attempts + 1):
        try:
            return fn()
        except RETRYABLE as exc:
            if attempt == attempts:
                raise                                                             # попытки кончились — ошибка наверх
            delay = min(cap, base * 2 ** (attempt - 1)) + random.uniform(0, 0.5)  # экспонента + случайный сдвиг
            print(f"попытка {attempt} не удалась: {type(exc).__name__}; жду {delay:.1f}s", file=sys.stderr)
            time.sleep(delay)

In [ ]:
client = make_client(timeout=TIMEOUT)
answer = with_retries(
    lambda: client.chat.completions.create(          # lambda: сам запрос упакован в функцию без аргументов
        model=MODEL,
        max_tokens=60,
        messages=[{"role": "user", "content": PROMPT}],
    )
)
print(answer.choices[0].message.content)